<div align="center">

# Train a SLM with a BPE Tokenizer

## TRI AI Saturdays Cohort 10 - Lab Session
### Wednesday, July 8, 2026 | 7:00 - 9:00 PM WAT

**In partnership with Google DeepMind**

</div>

## Lab Overview

In this hands-on session, you will:

1. **Understand** the Byte-Pair Encoding (BPE) algorithm
2. **Implement** BPE from scratch in pure Python
3. **Use** Hugging Face's `tokenizers` library
4. **Experiment** with different vocabulary sizes
5. **Connect** tokenization to real-world AI applications

### Why BPE Matters
- First step in every NLP pipeline
- Impacts model quality, speed, and cost
- Custom tokenizers needed for African languages

## Setup & Imports

In [7]:
# Install if needed:
# !pip install tokenizers numpy matplotlib pandas

import re, math
from collections import defaultdict, Counter
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

try:
    from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
    HAS_TOKENIZERS = True
    print("tokenizers library ready!")
except ImportError:
    HAS_TOKENIZERS = False
    print("Install: !pip install tokenizers")
print("Setup complete.")

tokenizers library ready!
Setup complete.


## Part 1: Build BPE from Scratch

**Training phase:**
1. Start with text, split into characters (initial vocab)
2. Count adjacent pairs, merge the most frequent
3. Repeat until desired vocab size

**Encoding phase:**
1. Split new text into characters
2. Apply learned merges in order
3. Return subword tokens

In [8]:
# BPE helper functions

def get_vocab(corpus):
    vocab = {}
    for word in corpus.lower().split():
        tokenized = " ".join(list(word)) + " </w>"
        vocab[tokenized] = vocab.get(tokenized, 0) + 1
    return vocab

def get_stats(vocab):
    pairs = defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i + 1])] += freq
    return pairs

def merge_vocab(pair, vocab):
    a, b = pair
    merged = a + b.replace("</w>", "")
    pattern = " ".join([a, b])
    new_vocab = {}
    for word, freq in vocab.items():
        new_vocab[word.replace(pattern, merged)] = freq
    return new_vocab

def bpe_train(corpus, vocab_size, verbose=True):
    vocab = get_vocab(corpus)
    initial_size = len(set(" ".join(vocab.keys()).split()))
    num_merges = vocab_size - initial_size
    merges = []
    if verbose:
        print(f"Initial symbols: {initial_size}, Target: {vocab_size}")
    for i in range(num_merges):
        pairs = get_stats(vocab)
        if not pairs:
            break
        best = max(pairs, key=pairs.get)
        vocab = merge_vocab(best, vocab)
        merges.append(best)
        if verbose and (i < 10 or (i+1) % 10 == 0):
            print(f"  Merge {i+1}: '{best[0]}' + '{best[1]}' -> '{best[0]+best[1]}'")
    if verbose:
        print(f"Done. {len(merges)} merges performed.")
    return merges, vocab, initial_size

def tokenize_bpe(text, merges):
    words = text.lower().split()
    tokens = []
    for word in words:
        symbols = list(word) + ["</w>"]
        for a, b in merges:
            i = 0
            while i < len(symbols) - 1:
                if symbols[i] == a and symbols[i+1] == b:
                    symbols = symbols[:i] + [a+b] + symbols[i+2:]
                else:
                    i += 1
        tokens.extend(symbols[:-1])
    return tokens

print("BPE functions defined.")

BPE functions defined.


## Part 2: Train BPE on an Agriculture Corpus

In [9]:
CORPUS = (
    "Cassava is a major food crop in Nigeria and across sub-Saharan Africa. "
    "Farmers in Kenya grow maize beans and sweet potatoes. "
    "The cocoa industry in Ghana and Ivory Coast produces most of the world's chocolate. "
    "Climate change affects rainfall patterns across East Africa. "
    "Drought resistant crops like millet and sorghum are important for food security. "
    "Smallholder farmers use sustainable farming practices to improve crop yields. "
    "Mobile technology helps farmers access weather forecasts and market prices. "
    "Cassava leaves are also nutritious and used in soups across West Africa. "
    "Kenyan tea and Ethiopian coffee are famous export crops. "
    "Soil health is critical for agricultural productivity in rural communities."
)

print("Corpus:", len(CORPUS.split()), "words")

merges, final_vocab, initial_size = bpe_train(CORPUS, vocab_size=50, verbose=True)

Corpus: 106 words
Initial symbols: 29, Target: 50
  Merge 1: 's' + '</w>' -> 's</w>'
  Merge 2: 'a' + 'n' -> 'an'
  Merge 3: 'e' + '</w>' -> 'e</w>'
  Merge 4: 'r' + 'o' -> 'ro'
  Merge 5: 't' + '</w>' -> 't</w>'
  Merge 6: 'd' + '</w>' -> 'd</w>'
  Merge 7: '.' + '</w>' -> '.</w>'
  Merge 8: 'o' + 'r' -> 'or'
  Merge 9: 'i' + 'n' -> 'in'
  Merge 10: 'r' + 'i' -> 'ri'
  Merge 20: 'a' + 'l' -> 'al'
Done. 21 merges performed.


### Viewing Learned Merges

In [10]:
merge_df = pd.DataFrame(merges, columns=["S1", "S2"])
merge_df["Merged"] = merge_df["S1"] + merge_df["S2"].str.replace("</w>", "")
merge_df["#"] = range(1, len(merges)+1)
print(merge_df[["#", "S1", "S2", "Merged"]].to_string(index=False))

 # S1   S2 Merged
 1  s </w>      s
 2  a    n     an
 3  e </w>      e
 4  r    o     ro
 5  t </w>      t
 6  d </w>      d
 7  . </w>      .
 8  o    r     or
 9  i    n     in
10  r    i     ri
11  a    r     ar
12  e    s     es
13  a </w>      a
14  c    a     ca
15  c   ro    cro
16 an    d    and
17  s    t     st
18  e    r     er
19  c    o     co
20  a    l     al
21  s    s     ss


### Tokenizing New Text

In [11]:
test_sentences = [
    "Farmers grow cassava in Nigeria",
    "Coffee from Ethiopia is delicious",
    "Climate change affects maize yields"
]
for s in test_sentences:
    toks = tokenize_bpe(s, merges)
    print(f"'{s}'")
    print(f"Tokens: {toks}  |  Count: {len(tokens)}\n")

'Farmers grow cassava in Nigeria'


NameError: name 'tokens' is not defined

## Part 3: Hugging Face tokenizers Library

In [ ]:
if HAS_TOKENIZERS:
    t = Tokenizer(models.BPE())
    t.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=True)
    trainer = trainers.BpeTrainer(
        vocab_size=1000, min_frequency=2,
        special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"],
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet()
    )
    t.train_from_iterator([CORPUS], trainer=trainer)
    t.decoder = decoders.ByteLevel()
    print(f"Vocab size: {t.get_vocab_size()}")
else:
    print("tokenizers library not available")

### Testing Our Trained Tokenizer

In [ ]:
if HAS_TOKENIZERS:
    tests = [
        "Cassava farming in Nigeria",
        "Coffee and tea are important exports",
        "Climate resilient crops for Africa"
    ]
    for text in tests:
        enc = t.encode(text)
        print(f"'{text}'")
        print(f"Tokens: {enc.tokens}")
        print(f"Decoded: {t.decode(enc.ids)}\n")

## Part 4: Experiment with Vocabulary Size

In [ ]:
if HAS_TOKENIZERS:
    vocab_sizes = [50, 100, 200, 500, 1000]
    test = "Cassava farmers in Nigeria use mobile technology for weather forecasts"
    print(f"Test sentence: '{test}'\n")
    results = []
    for size in vocab_sizes:
        tok = Tokenizer(models.BPE())
        tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=True)
        tr = trainers.BpeTrainer(vocab_size=size, min_frequency=1,
                                 special_tokens=["[UNK]"],
                                 initial_alphabet=pre_tokenizers.ByteLevel.alphabet())
        tok.train_from_iterator([CORPUS], trainer=tr)
        tok.decoder = decoders.ByteLevel()
        enc = tok.encode(test)
        results.append({"Vocab": size, "Tokens": len(enc.tokens)})
        print(f"Vocab={size:4d} -> {len(enc.tokens):2d} tokens -> {enc.tokens}")

    import matplotlib.pyplot as plt
    plt.figure(figsize=(8, 4))
    plt.plot([r["Vocab"] for r in results], [r["Tokens"] for r in results],
             "bo-", linewidth=2, markersize=8)
    plt.xlabel("Vocabulary Size")
    plt.ylabel("Number of Tokens")
    plt.title("Vocabulary Size vs Token Count")
    plt.grid(alpha=0.3)
    plt.show()
    print("Larger vocab = fewer tokens, but more model parameters.")

## Part 5: Healthcare Application

In [ ]:
if HAS_TOKENIZERS:
    medical = [
        "Malaria remains a leading cause of death in sub-Saharan Africa",
        "The patient presents with fever chills and headache",
        "Artemisinin combination therapy is the recommended treatment",
        "Maternal mortality rates have declined in Rwanda"
    ]
    combined = CORPUS + " " + " ".join(medical)
    t_med = Tokenizer(models.BPE())
    t_med.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=True)
    tr = trainers.BpeTrainer(vocab_size=200, min_frequency=1,
                             special_tokens=["[UNK]"],
                             initial_alphabet=pre_tokenizers.ByteLevel.alphabet())
    t_med.train_from_iterator([combined], trainer=tr)
    t_med.decoder = decoders.ByteLevel()
    for text in medical:
        enc = t_med.encode(text)
        print(f"{text}")
        print(f"-> Tokens: {enc.tokens}\n")

## Part 6: Your Turn!

**Challenge A:** Tokenize this Yoruba farming text.
**Challenge B:** Compare vocab sizes 100, 300, 500 on the same sentence.
**Challenge C:** Load a real tokenizer (Gemma, GPT-2) and analyze how it tokenizes your name, country, and local food.
**Challenge D:** Take a sentence from your project and tokenize with 3 different vocab sizes.

In [ ]:
# Challenge A starter
yoruba = "Awon agbe n gbin isu, agbado, ati ege ni orile-ede Naijiria."
tokens = tokenize_bpe(yoruba, merges)
print(f"Yoruba text: {yoruba}")
print(f"Tokens: {tokens}")
print(f"Chars: {len(yoruba)}, Tokens: {len(tokens)}")

## Key Takeaways

1. **BPE** iteratively merges frequent character pairs into a subword vocabulary.
2. **Vocab size trade-off:** Small = handles rare words; Large = efficient but more params.
3. **Custom tokenizers** are needed for non-English and specialized domains.
4. **Tokenization** is the first step in every NLP pipeline.

**Next Session: Saturday, July 11 - Embeddings!**

---

*TRI AI Saturdays Cohort 10 | In partnership with Google DeepMind*